# ============================================
#  Notebook 04c — RAG Question Generation & Evaluation
#  Memorial Sloan Kettering | Goel Lab
# ============================================

# Notebook 04c: RAG Evaluation Dataset Generation

**Purpose:**  
Generate a feature-aware question dataset from `feature_document_context.py` and evaluate  
OpenAI's `file_search` RAG pipeline on per-case consolidated clinical documents.

**Pipeline:**
1. Generate template questions (one per feature, from extraction hints)
2. Generate case-specific questions (GPT-4o, one per case × feature)
3. Upload consolidated `.txt` files to an OpenAI Vector Store
4. Query vector store for each question; retrieve top-k case documents
5. Evaluate: Recall@k, Precision@k, MRR, MAP — overall and stratified by feature / domain / fabrication risk

**Inputs (from NB04b):**
- `DATA_PRIVATE_DIR/extracted_text_consolidated/*.txt` — one file per patient case  
- `data/processed/patient_case_manifest.csv`  

**Outputs:**
- `data/processed/rag_question_dataset.csv` — question dataset (committed, no PHI)
- `data/processed/rag_evaluation_results.csv` — per-question retrieval metrics
- `reports/rag_evaluation_summary.png` — metric plots by feature and domain
- `data/processed/vector_store_config.json` — vector store ID for reuse

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import concurrent

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
sns.set_style('whitegrid')

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT     = Path(os.getenv("PROJECT_ROOT",
    "/Users/robertjames/Documents/GitHub/llm_summarization_br_ca"))
DATA_PRIVATE_DIR = Path(os.getenv("DATA_PRIVATE_DIR", "/Users/robertjames/data_private"))

CONSOLIDATED_DIR = DATA_PRIVATE_DIR / "extracted_text_consolidated"
PROCESSED_DIR    = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR       = PROJECT_ROOT / "reports"

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from llm_eval_by_llm.feature_document_context import (
    FEATURE_DOCUMENT_CONTEXT,
    FEATURE_SECTIONS,
    EXTRACTION_HINTS,
    HIGH_FABRICATION_RISK,
    filter_text_to_relevant_sections,
)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print(f"Features loaded  : {len(FEATURE_DOCUMENT_CONTEXT)}")
print(f"Consolidated dir : {CONSOLIDATED_DIR}")
print(f"Consolidated files: {len(list(CONSOLIDATED_DIR.glob('*.txt')))}")

---
## Part 1: Template Question Generation (Feature-Level)

One deterministic question per feature, derived directly from `extraction_hint` and `expected_values`.  
These questions are **domain-agnostic** — they describe *what to look for* without naming a specific case.  
Used as a sanity-check baseline for RAG retrieval.

In [ ]:
# Clinical extraction template structure from the structured radiology/pathology form
# (matches images provided: Mammogram → US → MRI → Pathology section hierarchy)
CLINICAL_TEMPLATE_CONTEXT = """
Instruction to LLM: Extract information from radiology and pathology reports
according to the structure below. Present findings chronologically.

1. Radiology Reports

Mammogram (Date, screening or diagnostic study)
- Laterality
- Abnormality (e.g., asymmetry, calcifications, distortion, mass)
- Size: cm
- Location: (quadrant, clock face, depth)
- Interval change
- Recommendation

Ultrasound (Date)
- Laterality
- Location: (quadrant, clock face, distance from nipple)
- Lesion size: ___ x ___ x ___ cm
- Morphology: (solid, cystic, complex, irregular, etc.)
- Findings correlate with Mammogram abnormalities
- Lymph node findings
- Recommendation

MRI (Date)
- Findings Laterality
- Location: (quadrant, clock face, depth)
- Abnormality type
  - Mass (shape, margins, enhancement, size)
  - Non-mass enhancement (distribution, pattern)
- Associated features (skin thickening, nipple retraction, chest wall involvement,
  multifocality/multicentricity)
- Findings correlate with MMG and US abnormalities
"""

def generate_template_question(feature_key: str, ctx: dict) -> str:
    """Generate a deterministic template question from extraction hint + expected values."""
    hint      = ctx["extraction_hint"][:600]
    expected  = ctx["expected_values"][:300]
    sections  = ", ".join(ctx["primary_sections"])
    prompt = (
        f"You are creating evaluation questions for a clinical RAG pipeline.\n\n"
        f"Clinical extraction template:\n{CLINICAL_TEMPLATE_CONTEXT}\n\n"
        f"Feature: {ctx['display']}\n"
        f"Relevant document sections: {sections}\n"
        f"What to look for: {hint}\n"
        f"Expected value format: {expected}\n\n"
        f"Generate ONE precise clinical question that:\n"
        f"1. Can only be answered by reading the {sections} section(s) of a patient's source documents\n"
        f"2. Would require retrieving the correct patient case document to answer\n"
        f"3. Is specific enough that it cannot be answered from general medical knowledge alone\n"
        f"4. Asks for the exact value as documented (not an interpretation)\n"
        f"Output only the question text, no preamble."
    )
    response = client.responses.create(
        input=prompt,
        model="gpt-4o",
    )
    return response.output[0].content[0].text.strip()

print("Generating template questions for all 14 features...")
template_questions = {}
for key, ctx in tqdm(FEATURE_DOCUMENT_CONTEXT.items(), desc="Template Qs"):
    template_questions[key] = generate_template_question(key, ctx)

print("\nTemplate questions:")
for key, q in template_questions.items():
    print(f"  [{key}]\n    {q}\n")

---
## Part 2: Case-Specific Question Generation (GPT-4o)

For each patient case × feature, GPT-4o reads the **relevant section(s)** of the consolidated  
text (filtered via `filter_text_to_relevant_sections`) and generates a question whose answer  
is documented in that specific case file.

This mirrors the OpenAI cookbook approach but scoped per-feature, so each question:  
- Can **only** be answered from the correct patient case document  
- Tests whether retrieval surfaces the right case for that feature

In [ ]:
SECTION_HEADER_MAP = {
    "hpi":       "=== HPI / CLINICAL NOTES ===",
    "radiology": "=== RADIOLOGY REPORTS ===",
    "pathology": "=== PATHOLOGY REPORTS ===",
    "genetics":  "=== GENETICS / MOLECULAR ===",
}

def generate_case_feature_question(
    patient_case_id: str,
    feature_key: str,
    ctx: dict,
    case_text_filtered: str,
) -> dict:
    """
    Generate one case-specific question for a given feature.
    Returns a dict with question, expected_file, feature metadata.
    """
    if not case_text_filtered.strip():
        return None

    hint     = ctx["extraction_hint"][:500]
    expected = ctx["expected_values"][:250]
    sections = ", ".join(ctx["primary_sections"])

    prompt = (
        f"You are creating evaluation questions for a clinical RAG retrieval system.\n\n"
        f"Clinical extraction template:\n{CLINICAL_TEMPLATE_CONTEXT}\n\n"
        f"The following is the {sections} section from ONE specific patient's clinical record:\n"
        f"---\n{case_text_filtered[:3000]}\n---\n\n"
        f"Feature to extract: {ctx['display']}\n"
        f"Extraction guidance: {hint}\n"
        f"Expected value format: {expected}\n\n"
        f"Generate ONE specific question about this patient's {ctx['display']} that:\n"
        f"1. Can ONLY be answered using THIS patient's document (not general medical knowledge)\n"
        f"2. Has a clear, documentable answer present in the text above\n"
        f"3. Would require retrieving this specific patient's file to answer correctly\n"
        f"4. Asks for a factual value as reported, not an interpretation\n"
        f"Output only the question text, no preamble."
    )

    try:
        response = client.responses.create(
            input=prompt,
            model="gpt-4o",
        )
        question = response.output[0].content[0].text.strip()
    except Exception as e:
        print(f"  Error generating Q for {patient_case_id}/{feature_key}: {e}")
        return None

    return {
        "patient_case_id":      patient_case_id,
        "expected_file":        f"{patient_case_id}.txt",
        "feature_key":          feature_key,
        "feature_display":      ctx["display"],
        "domain":               ctx["domain"],
        "primary_sections":     "|".join(ctx["primary_sections"]),
        "fabrication_risk":     ctx["fabrication_risk"],
        "question":             question,
        "context_chars":        len(case_text_filtered),
        "question_type":        "case_specific",
    }

print("Ready to generate case-specific questions.")
print(f"Consolidated cases: {len(list(CONSOLIDATED_DIR.glob('*.txt')))}")
print(f"Features: {len(FEATURE_DOCUMENT_CONTEXT)}")
print(f"Max questions: {len(list(CONSOLIDATED_DIR.glob('*.txt'))) * len(FEATURE_DOCUMENT_CONTEXT)}")

In [ ]:
# Generate case-specific questions
# Runs in parallel per case (sequential per feature within each case to respect rate limits)
consolidated_files = sorted(CONSOLIDATED_DIR.glob("*.txt"))

question_rows = []

def process_case(txt_path: Path) -> list:
    patient_case_id = txt_path.stem
    full_text = txt_path.read_text(encoding="utf-8")
    rows = []
    for feature_key, ctx in FEATURE_DOCUMENT_CONTEXT.items():
        filtered = filter_text_to_relevant_sections(
            full_text, feature_key,
            section_header_map=SECTION_HEADER_MAP
        )
        result = generate_case_feature_question(
            patient_case_id, feature_key, ctx, filtered
        )
        if result:
            rows.append(result)
    return rows

if not consolidated_files:
    print("No consolidated files found. Run NB04 + NB04b first.")
else:
    # Parallel across cases, max 5 workers to respect OpenAI rate limits
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(process_case, f): f for f in consolidated_files}
        for future in tqdm(
            concurrent.futures.as_completed(futures),
            total=len(consolidated_files),
            desc="Generating case questions"
        ):
            result = future.result()
            if result:
                question_rows.extend(result)

    # Add template questions (one row per feature, no specific case)
    for feature_key, question in template_questions.items():
        ctx = FEATURE_DOCUMENT_CONTEXT[feature_key]
        question_rows.append({
            "patient_case_id":  None,
            "expected_file":    None,
            "feature_key":      feature_key,
            "feature_display":  ctx["display"],
            "domain":           ctx["domain"],
            "primary_sections": "|".join(ctx["primary_sections"]),
            "fabrication_risk": ctx["fabrication_risk"],
            "question":         question,
            "context_chars":    0,
            "question_type":    "template",
        })

    df_questions = pd.DataFrame(question_rows)
    df_questions.to_csv(PROCESSED_DIR / "rag_question_dataset.csv", index=False)
    print(f"\nTotal questions  : {len(df_questions)}")
    print(f"  Case-specific  : {(df_questions['question_type']=='case_specific').sum()}")
    print(f"  Template       : {(df_questions['question_type']=='template').sum()}")
    print(f"Saved → {PROCESSED_DIR / 'rag_question_dataset.csv'}")
    df_questions.head(4)

---
## Part 3: Create OpenAI Vector Store and Upload Consolidated Files

Each patient case's consolidated `.txt` is uploaded as a separate file.  
OpenAI chunks, embeds, and indexes it automatically via `file_search`.

The vector store ID is saved to `data/processed/vector_store_config.json` for reuse.

In [ ]:
VS_CONFIG_PATH = PROCESSED_DIR / "vector_store_config.json"

def create_vector_store(store_name: str) -> dict:
    vector_store = client.vector_stores.create(name=store_name)
    details = {
        "id":         vector_store.id,
        "name":       vector_store.name,
        "created_at": vector_store.created_at,
        "file_count": vector_store.file_counts.completed,
    }
    print(f"Vector store created: {details}")
    return details

def upload_single_txt(file_path: Path, vector_store_id: str) -> dict:
    try:
        file_response = client.files.create(
            file=(file_path.name, file_path.read_bytes(), "text/plain"),
            purpose="assistants"
        )
        client.vector_stores.files.create(
            vector_store_id=vector_store_id,
            file_id=file_response.id
        )
        return {"file": file_path.name, "file_id": file_response.id, "status": "success"}
    except Exception as e:
        print(f"  Error uploading {file_path.name}: {e}")
        return {"file": file_path.name, "status": "failed", "error": str(e)}

def upload_consolidated_files(vector_store_id: str) -> dict:
    txt_files = sorted(CONSOLIDATED_DIR.glob("*.txt"))
    stats = {"total": len(txt_files), "success": 0, "failed": 0, "errors": [], "file_map": {}}
    print(f"{len(txt_files)} consolidated case files to upload...")

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {
            executor.submit(upload_single_txt, f, vector_store_id): f
            for f in txt_files
        }
        for future in tqdm(
            concurrent.futures.as_completed(futures), total=len(txt_files)
        ):
            result = future.result()
            if result["status"] == "success":
                stats["success"] += 1
                stats["file_map"][result["file"]] = result["file_id"]
            else:
                stats["failed"] += 1
                stats["errors"].append(result)
    return stats

# Create or reuse vector store
if VS_CONFIG_PATH.exists():
    vs_details = json.loads(VS_CONFIG_PATH.read_text())
    print(f"Reusing existing vector store: {vs_details['id']}")
else:
    vs_details = create_vector_store("msk_goel_lab_clinical_cases")
    upload_stats = upload_consolidated_files(vs_details["id"])
    vs_details["upload_stats"] = upload_stats
    VS_CONFIG_PATH.write_text(json.dumps(vs_details, indent=2))
    print(f"\nUpload complete: {upload_stats['success']}/{upload_stats['total']} files")
    print(f"Config saved → {VS_CONFIG_PATH}")

VECTOR_STORE_ID = vs_details["id"]
print(f"\nVector store ID: {VECTOR_STORE_ID}")

---
## Part 4: Query Vector Store and Evaluate Retrieval

For each case-specific question:  
- Query the vector store with `file_search` (top-k = 5)  
- Check whether `{patient_case_id}.txt` appears in retrieved files  
- Calculate per-query: correct@k, reciprocal rank, average precision

In [ ]:
K = 5  # top-k for retrieval evaluation

def query_vector_store(question: str, expected_file: str, k: int = K) -> dict:
    """
    Send question to file_search via Responses API.
    Returns retrieval metrics for this question.
    """
    try:
        response = client.responses.create(
            input=question,
            model="gpt-4o-mini",
            tools=[{
                "type": "file_search",
                "vector_store_ids": [VECTOR_STORE_ID],
                "max_num_results": k,
            }],
            tool_choice="required",
        )
    except Exception as e:
        return {"error": str(e), "correct": False, "rr": 0.0, "avg_precision": 0.0,
                "retrieved_files": [], "rank": None}

    # Extract retrieved filenames from annotations
    annotations = None
    for output_item in response.output:
        if hasattr(output_item, 'content') and output_item.content:
            for content_item in output_item.content:
                if hasattr(content_item, 'annotations') and content_item.annotations:
                    annotations = content_item.annotations
                    break

    if not annotations:
        return {"error": "no_annotations", "correct": False, "rr": 0.0,
                "avg_precision": 0.0, "retrieved_files": [], "rank": None}

    retrieved_files = [a.filename for a in annotations[:k]]

    # Metrics
    correct = expected_file in retrieved_files
    rank    = (retrieved_files.index(expected_file) + 1) if correct else None
    rr      = (1 / rank) if rank else 0.0

    precisions  = []
    n_relevant  = 0
    for i, fname in enumerate(retrieved_files):
        if fname == expected_file:
            n_relevant += 1
            precisions.append(n_relevant / (i + 1))
    avg_precision = float(np.mean(precisions)) if precisions else 0.0

    return {
        "correct":          correct,
        "rank":             rank,
        "rr":               rr,
        "avg_precision":    avg_precision,
        "retrieved_files":  "|".join(retrieved_files),
        "error":            None,
    }

print(f"Evaluation config: k={K}, model=gpt-4o-mini, vector_store={VECTOR_STORE_ID}")

In [ ]:
# Run evaluation on case-specific questions only
eval_rows = df_questions[df_questions["question_type"] == "case_specific"].copy()
eval_rows = eval_rows.dropna(subset=["expected_file"]).reset_index(drop=True)

print(f"Evaluating {len(eval_rows)} case-specific questions...")

def process_eval_row(row):
    result = query_vector_store(row["question"], row["expected_file"], k=K)
    return {**row.to_dict(), **result}

eval_results = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {
        executor.submit(process_eval_row, row): i
        for i, row in eval_rows.iterrows()
    }
    for future in tqdm(
        concurrent.futures.as_completed(futures),
        total=len(eval_rows),
        desc="Evaluating RAG"
    ):
        eval_results.append(future.result())

df_eval = pd.DataFrame(eval_results)
df_eval.to_csv(PROCESSED_DIR / "rag_evaluation_results.csv", index=False)
print(f"\nResults saved → {PROCESSED_DIR / 'rag_evaluation_results.csv'}")
df_eval[["feature_key", "patient_case_id", "correct", "rank", "rr", "avg_precision"]].head(8)

---
## Part 5: Aggregate Metrics — Overall and Stratified

In [ ]:
# ── Overall metrics ───────────────────────────────────────────────────────────
ok = df_eval[df_eval["error"].isna()]
n  = len(ok)

recall_k    = ok["correct"].mean()
mrr         = ok["rr"].mean()
map_score   = ok["avg_precision"].mean()

print(f"Overall metrics (k={K}, n={n} questions):")
print(f"  Recall@{K}   : {recall_k:.4f}")
print(f"  Precision@{K}: {recall_k:.4f}")
print(f"  MRR         : {mrr:.4f}")
print(f"  MAP         : {map_score:.4f}")

# ── Per-feature metrics ───────────────────────────────────────────────────────
feature_metrics = (
    ok.groupby(["feature_key", "feature_display", "domain", "fabrication_risk"])
    .agg(
        n_questions=("correct", "count"),
        recall_at_k=("correct", "mean"),
        mrr=("rr", "mean"),
        map=("avg_precision", "mean"),
    )
    .reset_index()
    .sort_values("recall_at_k", ascending=False)
)

print(f"\nPer-feature Recall@{K}:")
print(feature_metrics[["feature_display", "domain", "fabrication_risk",
                        "recall_at_k", "mrr", "map"]].to_string(index=False))

# ── Per-domain metrics ────────────────────────────────────────────────────────
domain_metrics = (
    ok.groupby("domain")
    .agg(recall_at_k=("correct", "mean"), mrr=("rr", "mean"), map=("avg_precision", "mean"),
         n=("correct", "count"))
    .reset_index()
)
print(f"\nPer-domain metrics:")
print(domain_metrics.to_string(index=False))

# ── By fabrication risk ───────────────────────────────────────────────────────
fab_metrics = (
    ok.groupby("fabrication_risk")
    .agg(recall_at_k=("correct", "mean"), mrr=("rr", "mean"), map=("avg_precision", "mean"),
         n=("correct", "count"))
    .reset_index()
)
print(f"\nBy fabrication risk:")
print(fab_metrics.to_string(index=False))

In [ ]:
# ── Visualization ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle(f"RAG Evaluation — Feature-Aware Retrieval (k={K})",
             fontsize=16, fontweight="bold")

FAB_COLORS = {"high": "#e74c3c", "medium": "#f39c12", "low": "#2ecc71"}
DOMAIN_COLORS = {"radiology": "#3498db", "pathology": "#e74c3c",
                  "hpi": "#2ecc71", "mixed": "#9b59b6"}

# 1. Recall@k per feature (horizontal bar, coloured by fabrication risk)
fm = feature_metrics.sort_values("recall_at_k")
colors = [FAB_COLORS.get(r, "#95a5a6") for r in fm["fabrication_risk"]]
axes[0, 0].barh(fm["feature_display"], fm["recall_at_k"], color=colors, edgecolor="black")
axes[0, 0].axvline(recall_k, color="black", linestyle="--", linewidth=1, label=f"Overall={recall_k:.2f}")
axes[0, 0].set_xlabel(f"Recall@{K}")
axes[0, 0].set_title(f"Recall@{K} by Feature\n(red=high fab risk)", fontweight="bold")
axes[0, 0].legend(fontsize=8)
axes[0, 0].tick_params(axis="y", labelsize=8)

# 2. MRR per feature
fm2 = feature_metrics.sort_values("mrr")
axes[0, 1].barh(fm2["feature_display"], fm2["mrr"],
                color=[FAB_COLORS.get(r, "#95a5a6") for r in fm2["fabrication_risk"]],
                edgecolor="black")
axes[0, 1].axvline(mrr, color="black", linestyle="--", linewidth=1, label=f"Overall={mrr:.2f}")
axes[0, 1].set_xlabel("MRR")
axes[0, 1].set_title("Mean Reciprocal Rank by Feature", fontweight="bold")
axes[0, 1].legend(fontsize=8)
axes[0, 1].tick_params(axis="y", labelsize=8)

# 3. MAP per feature
fm3 = feature_metrics.sort_values("map")
axes[0, 2].barh(fm3["feature_display"], fm3["map"],
                color=[FAB_COLORS.get(r, "#95a5a6") for r in fm3["fabrication_risk"]],
                edgecolor="black")
axes[0, 2].axvline(map_score, color="black", linestyle="--", linewidth=1, label=f"Overall={map_score:.2f}")
axes[0, 2].set_xlabel("MAP")
axes[0, 2].set_title("Mean Average Precision by Feature", fontweight="bold")
axes[0, 2].legend(fontsize=8)
axes[0, 2].tick_params(axis="y", labelsize=8)

# 4. Metrics by domain
x = np.arange(len(domain_metrics))
w = 0.25
axes[1, 0].bar(x - w, domain_metrics["recall_at_k"], w, label=f"Recall@{K}", color="#3498db", edgecolor="black")
axes[1, 0].bar(x,     domain_metrics["mrr"],         w, label="MRR",         color="#2ecc71", edgecolor="black")
axes[1, 0].bar(x + w, domain_metrics["map"],         w, label="MAP",         color="#e74c3c", edgecolor="black")
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(domain_metrics["domain"], rotation=15)
axes[1, 0].set_title("Retrieval Metrics by Domain", fontweight="bold")
axes[1, 0].legend(fontsize=8)
axes[1, 0].set_ylim(0, 1.05)

# 5. Metrics by fabrication risk
x2 = np.arange(len(fab_metrics))
axes[1, 1].bar(x2 - w, fab_metrics["recall_at_k"], w, label=f"Recall@{K}", color="#3498db", edgecolor="black")
axes[1, 1].bar(x2,     fab_metrics["mrr"],         w, label="MRR",         color="#2ecc71", edgecolor="black")
axes[1, 1].bar(x2 + w, fab_metrics["map"],         w, label="MAP",         color="#e74c3c", edgecolor="black")
axes[1, 1].set_xticks(x2)
axes[1, 1].set_xticklabels(fab_metrics["fabrication_risk"])
axes[1, 1].set_title("Retrieval Metrics by Fabrication Risk", fontweight="bold")
axes[1, 1].legend(fontsize=8)
axes[1, 1].set_ylim(0, 1.05)

# 6. Distribution of reciprocal ranks
axes[1, 2].hist(ok["rr"], bins=[0, 0.2, 0.34, 0.5, 1.01],
                color="#9b59b6", edgecolor="black", rwidth=0.8)
axes[1, 2].set_xticks([0.1, 0.27, 0.42, 0.75])
axes[1, 2].set_xticklabels(["Not found", "Rank 3", "Rank 2", "Rank 1"])
axes[1, 2].set_ylabel("Number of Questions")
axes[1, 2].set_title("Retrieval Rank Distribution", fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "rag_evaluation_summary.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved → {OUTPUT_DIR / 'rag_evaluation_summary.png'}")

---
## Part 6: Failure Analysis

Inspect questions where the correct case file was NOT retrieved — to identify  
whether failures cluster around specific features, fabrication-risk levels, or case types.

In [ ]:
failures = df_eval[df_eval["correct"] == False].copy()

print(f"Failed retrievals: {len(failures)} / {len(ok)} ({len(failures)/len(ok)*100:.1f}%)")
print()

if len(failures) > 0:
    print("Failures by feature:")
    print(failures["feature_display"].value_counts().to_string())
    print()
    print("Failures by fabrication risk:")
    print(failures["fabrication_risk"].value_counts().to_string())
    print()
    print("Sample failures:")
    for _, row in failures.head(5).iterrows():
        print(f"  Case     : {row['patient_case_id']}")
        print(f"  Feature  : {row['feature_display']}")
        print(f"  Question : {row['question']}")
        print(f"  Expected : {row['expected_file']}")
        print(f"  Retrieved: {row['retrieved_files']}")
        print()
else:
    print("No retrieval failures.")

---
## Summary of Outputs

| Output | Path | Committed? |
|--------|------|------------|
| Question dataset | `data/processed/rag_question_dataset.csv` | Yes |
| Per-question retrieval results | `data/processed/rag_evaluation_results.csv` | Yes |
| Vector store config (ID) | `data/processed/vector_store_config.json` | Yes |
| Evaluation plots | `reports/rag_evaluation_summary.png` | Yes |

**Key metrics reported:** Recall@k, Precision@k, MRR, MAP — overall + per feature + per domain + per fabrication risk level.

**Next step:** Use `VECTOR_STORE_ID` and `rag_question_dataset.csv` in NB09 (mCodeGPT feature extraction)  
to ground LLM extraction in retrieved context rather than full consolidated text.